# 2h. Multi-Session SBI: GP-Linked Trajectory Recovery

**Purpose**: Can the SBIFitter with GP-linked priors recover a known
parameter trajectory across sessions?

This is the prerequisite for 4b's GP-linked SBI section. If the
trajectory recovery fails here on synthetic data, the multi-session
SBI approach cannot be trusted on real data.

**Protocol**:
1. Define ground-truth η trajectory (smooth, GP-like)
2. Simulate multi-session data with varying η
3. Fit with SBIFitter using GPSpec-linked priors
4. Compare recovered vs true trajectory

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_NOTEBOOK_ROOT = _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _NOTEBOOK_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
if str(_NOTEBOOK_ROOT) not in sys.path:
	sys.path.insert(0, str(_NOTEBOOK_ROOT))

from shared_setup import *

import time
from models.BE_core import BEParams, BEModel
from models.SC_core import SCParams, SCModel
from behav_utils.data.synthetic import generate_synthetic_animal
from analysis.validation import generate_session_with_distribution
from behav_utils.data.structures import AnimalData, FittingData
from behav_utils.data.selection import fitting_data_from_sessions

SBI_OK = False
try:
    import torch
    from inference.fitting import SBIFitter, SBIResult
    from inference.types import GPSpec, ConstantSpec
    SBI_OK = True
    print(f'SBI available (torch={torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

## 1. Configuration

In [ ]:
QUICK = True

if QUICK:
    N_SESSIONS = 10
    TRIALS_PER_SESSION = 300
    BURN_IN = 500
    N_SBI_SIMS = 2_000
else:
    N_SESSIONS = 20
    TRIALS_PER_SESSION = 350
    BURN_IN = 1000
    N_SBI_SIMS = 30_000

SEED = 42
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

## 2. Define Ground-Truth Trajectory

A smooth η trajectory for BE: starts moderate, peaks during
"active learning", settles at expert level.

In [ ]:
# ── Ground-truth eta trajectory (hump-shaped) ──────────────────────────────
session_indices = np.arange(N_SESSIONS)

# Fixed perceptual params
SIGMA_PERCEP = 0.15
A_REPULSION = 0.10
ETA_RELAX = 0.12

# eta_learning trajectory: 0.10 -> 0.45 -> 0.15
frac = session_indices / max(N_SESSIONS - 1, 1)
true_eta = np.where(
    frac < 0.4,
    0.10 + (0.45 - 0.10) * (frac / 0.4),
    0.45 + (0.15 - 0.45) * ((frac - 0.4) / 0.6),
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(session_indices, true_eta, 'ko-', ms=5, label='True \u03b7')
ax.set_xlabel('Session')
ax.set_ylabel('\u03b7_learning')
ax.set_title('Ground-Truth Parameter Trajectory')
ax.legend()
plt.tight_layout()
plt.show()

print(f'eta range: [{true_eta.min():.3f}, {true_eta.max():.3f}]')

## 3. Simulate Multi-Session Data

In [ ]:
# ── Generate sessions with varying eta ─────────────────────────────────────
rng = np.random.default_rng(SEED)
sessions = []

for s in range(N_SESSIONS):
    params = BEParams(
        sigma_percep=SIGMA_PERCEP, A_repulsion=A_REPULSION,
        eta_learning=float(true_eta[s]), eta_relax=ETA_RELAX,
    )
    sim = BEModel.make_simulator(params, burn_in=BURN_IN, seed=SEED + s)
    sess = generate_session_with_distribution(
        session_idx=s, n_trials=TRIALS_PER_SESSION,
        distribution='uniform', animal_id='GP_test',
        simulator=sim, stage=STAGE, rng=rng,
    )
    sessions.append(sess)

# Check accuracy progression
accs = [s.stats(['accuracy'])['accuracy'] for s in sessions]
print(f'Accuracy: {accs[0]:.2f} -> {accs[N_SESSIONS//2]:.2f} -> {accs[-1]:.2f}')

## 4. Fit with GP-Linked SBI

Use SBIFitter with GPSpec for eta_learning (varies across sessions)
and ConstantSpec for the remaining parameters (fixed across sessions).

In [ ]:
GP_FITTED = False
recovered_eta = None

if not SBI_OK:
    print('SBI not available')
else:
    try:
        from models.BE_core import BEParams
        bounds = BEParams.get_bounds()

        fd = fitting_data_from_sessions(sessions, 'GP_test')

        fitter = SBIFitter(
            fitting_data=fd,
            model_type='be',
            param_links={
                'sigma_percep': ConstantSpec(bounds=bounds['sigma_percep']),
                'A_repulsion': ConstantSpec(bounds=bounds['A_repulsion']),
                'eta_learning': GPSpec(bounds=bounds['eta_learning']),
                'eta_relax': ConstantSpec(bounds=bounds['eta_relax']),
            },
            burn_in=BURN_IN,
        )

        print(f'Training SBIFitter ({N_SBI_SIMS} sims)...')
        t0 = time.time()
        result = fitter.train(n_simulations=N_SBI_SIMS, seed=SEED)
        elapsed = time.time() - t0
        print(f'Done in {elapsed:.1f}s')

        # Extract trajectories
        trajectories = fitter.extract_trajectories(result, n_samples=500)
        recovered_eta = trajectories.get('eta_learning', None)
        GP_FITTED = True

    except Exception as e:
        print(f'GP fitting failed: {e}')
        import traceback; traceback.print_exc()

## 5. Compare Recovered vs True Trajectory

In [ ]:
if GP_FITTED and recovered_eta is not None:
    # recovered_eta is a dict: mean, median, ci_low, ci_high, std, samples
    rec_median = np.asarray(recovered_eta['median'])
    rec_q05 = np.asarray(recovered_eta['ci_low'])
    rec_q95 = np.asarray(recovered_eta['ci_high'])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(session_indices, true_eta, 'ko-', ms=5, lw=2, label='True \u03b7')
    ax.plot(session_indices, rec_median, 's-', color='steelblue',
            ms=5, lw=2, label='Recovered median')
    ax.fill_between(session_indices, rec_q05, rec_q95,
                    color='steelblue', alpha=0.2, label='90% CI')
    ax.set_xlabel('Session')
    ax.set_ylabel('\u03b7_learning')
    ax.set_title('GP-Linked SBI: Trajectory Recovery')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Quantify
    rmse = np.sqrt(np.mean((true_eta - rec_median) ** 2))
    corr = np.corrcoef(true_eta, rec_median)[0, 1]
    print(f'RMSE: {rmse:.4f}')
    print(f'Correlation: {corr:.3f}')
    print(f'True range:      [{true_eta.min():.3f}, {true_eta.max():.3f}]')
    print(f'Recovered range: [{rec_median.min():.3f}, {rec_median.max():.3f}]')
    print(f'90% CI covers true: '
          f'{np.mean((true_eta >= rec_q05) & (true_eta <= rec_q95)):.0%} of sessions')
else:
    print('No GP-linked fit to compare')

## 6. Conclusion

**Pass criteria**:
- Correlation > 0.8 between true and recovered trajectory
- True trajectory falls within the 90% CI for most sessions
- Recovery captures the hump shape (peak, not just monotonic)

**If this fails**: The multi-session SBI approach (4b §7) cannot
be trusted. Use per-session grid-search fitting instead.

**QUICK mode caveat**: With 2k sims, the posterior will be rough.
Full run (30k sims) should give much better recovery.